    y      = E x                E  = [1, x_i, y_i]              (N x 3)
    E*     = (E^T E)^-1 E^T                                     (3 x N)
    y      = E_true alpha       E_true[i,m] = exp(i(k_m x_i + l_m y_i))
    x*     = E* E_true alpha = T alpha                          (3 x K)

    t0, tx, ty = rows of T
    H_x = tx / (i k)     R_x = |H_x|^2      -> 1 = faithfully reported
    H_y = ty / (i l)     R_y = |H_y|^2      -> 0 = invisible to the array

R is a PER-MODE ratio against the true derivative. R = 1 is good near the
origin (signal you want) and bad away from it (unresolved energy passing
straight through). The main lobe width is set by APERTURE; N controls how
well everything outside it is rejected.

In [1]:
import numpy as np

# ------------------------------------------------------------------ config params
A_KM = 10.0                      # square half-width
B_KM = A_KM * np.sqrt(2)         # shared circumradius, 14.142 km
LAT0, LON0 = 0.0, -140.0         # this is the center lat/lon of the stencil (doesn't matter)
DEG_LON_KM, DEG_LAT_KM = 111.320, 110.570 # convert deg to km
NX, NY, DX_DEG = 480, 360, 1 / 24        # tpose24 model domain: 20 x 15 deg at 1/24
SIGNAL_CUT_KM = 100.0            # lambda > this = signal, < this = leakage

In [2]:
def polygon(n, R, phi0_deg=0.0):
    """
    Regular n-gon on a circle of radius R, plus a center sample.
    Center sample doesn't affect the gradients but is realistic for the field.
    """
    th = np.deg2rad(phi0_deg) + 2 * np.pi * np.arange(n) / n
    return np.vstack([R * np.column_stack([np.cos(th), np.sin(th)]),
                      [[0.0, 0.0]]])

In [3]:
# More configuration choices -- we want to rotate a few of the shapes to see what happens
# Rotation choices: a regular N-gon repeats every 360/N degrees, and for ODD N
# a further rotation of 180/N maps it onto its own point-inversion, leaving |H|
# unchanged. So the distinct range is 360/N (even N) or 180/N (odd N).
ARRAYS = {
    "triangle":     polygon(3, B_KM, 0.0),
    "triangle 90":  polygon(3, B_KM, 90.0),
    "square":       np.array([[A_KM, A_KM], [-A_KM, A_KM],
                              [-A_KM, -A_KM], [A_KM, -A_KM], [0.0, 0.0]]),
    "diamond":      polygon(4, B_KM, 0.0),
    "hexagon":      polygon(6, B_KM, 0.0),
    "hexagon 90":   polygon(6, B_KM, 90.0),
    "octagon":      polygon(8, B_KM, 0.0),
    "octagon 22.5": polygon(8, B_KM, 22.5),
}
LABEL = {"triangle": "vertex east", "triangle 90": "vertex north",
         "square": f"$\\pm${A_KM:.0f}, $\\pm${A_KM:.0f} km",
         "diamond": "square rotated 45$\\degree$",
         "hexagon": "vertex on $+x$", "hexagon 90": "rotated 90$\\degree$",
         "octagon": "vertex on $+x$", "octagon 22.5": "rotated 22.5$\\degree$"}
CIRC = {k: B_KM for k in ARRAYS}

In [4]:
# need to identify the resolvable wavenumbers for the model domain
def model_wavenumbers():
    """(k, l) the model can represent. fx, fy in cyc/km; KX, KY in rad/m."""
    fx = np.fft.fftshift(np.fft.fftfreq(NX, d=DX_DEG)) / DEG_LON_KM
    fy = np.fft.fftshift(np.fft.fftfreq(NY, d=DX_DEG)) / DEG_LAT_KM
    FX, FY = np.meshgrid(fx, fy)
    return fx, fy, FX, FY, 2 * np.pi * FX / 1e3, 2 * np.pi * FY / 1e3


FX_C, FY_C, FX, FY, KX, KY = model_wavenumbers()
KABS = np.hypot(FX, FY)    # convert to cyc/km

In [5]:
def transfer(pts_km):
    """Return H_x, H_y (complex), R_x, R_y, and the noise amplification factor.

    Geometry only: no model field or spectrum enters, just the wavenumbers
    (KX, KY) at which the response is evaluated.
    """
    # x, y        : sample positions, m (input is km)
    x, y = pts_km[:, 0] * 1e3, pts_km[:, 1] * 1e3

    # E_geom      : (N, 3) plane-fit design matrix, columns [1, x, y]
    E_geom = np.column_stack([np.ones_like(x), x, y])

    # E_star      : (3, N) pseudo-inverse. Rows are the weight vectors
    #               g_0 [-], g_x [1/m], g_y [1/m]. inv() not pinv() so a
    #               collinear array raises rather than returning min-norm.
    E_star = np.linalg.inv(E_geom.T @ E_geom) @ E_geom.T

    # E_true      : (N, K) Fourier modes evaluated AT THE SAMPLE POINTS,
    #               exp(i(k x_i + l y_i)); K = number of model wavenumbers
    E_true = np.exp(1j * (np.outer(x, KX.ravel()) + np.outer(y, KY.ravel())))

    # T           : (3, K) fit coefficients per unit-amplitude mode
    #               rows = [mean, d/dx, d/dy]
    T = E_star @ E_true

    # tx, ty      : (ny, nx) fitted d/dx and d/dy per mode, 1/m
    tx, ty = T[1].reshape(KX.shape), T[2].reshape(KX.shape)

    # H_x, H_y    : dimensionless response, fitted derivative / true
    #               derivative (ik, il). 1 = faithful, 0 = invisible.
    with np.errstate(divide="ignore", invalid="ignore"):
        H_x, H_y = tx / (1j * KX), ty / (1j * KY)

    # kz, lz      : masks of the k = 0 and l = 0 lines, where the true
    #               derivative is zero and H is 0/0 or a pole
    kz, lz = KX == 0, KY == 0

    # gx_scale    : sum|g_x|, the magnitude scale of the x weights; used as
    # gy_scale      the reference for deciding whether tx, ty vanish
    gx_scale, gy_scale = np.abs(E_star[1]).sum(), np.abs(E_star[2]).sum()

    # lh_x, lh_y  : L'Hopital limit of H on those lines, valid only where the
    #               numerator vanishes (mirror-symmetric arrays); NaN
    #               elsewhere, where H truly diverges
    lh_x = np.exp(1j * np.outer(y, KY[kz])).T @ (E_star[1] * x)
    lh_y = np.exp(1j * np.outer(x, KX[lz])).T @ (E_star[2] * y)
    H_x[kz] = np.where(np.abs(tx[kz]) < 1e-10 * gx_scale, lh_x, np.nan)
    H_y[lz] = np.where(np.abs(ty[lz]) < 1e-10 * gy_scale, lh_y, np.nan)

    # namp        : noise amplification, 1/m. sigma_grad = namp * sigma_u.
    #               Equals 2/(rho sqrt(N)) for a regular N-gon.
    namp = np.sqrt((E_star[1] ** 2).sum() + (E_star[2] ** 2).sum())

    # R = |H|^2 is the power transfer function, so half power is R = 0.5
    return H_x, H_y, np.abs(H_x) ** 2, np.abs(H_y) ** 2, namp

In [6]:
import csv
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

In [7]:
R = {nm: transfer(p) for nm, p in ARRAYS.items()}


In [ ]:
# --------------------------------------------------------- R(k,l) figures
def comp_fig(idx, comp, fname):
    n = len(ARRAYS)
    fig, axs = plt.subplots(3, n, figsize=(3.35 * n, 11.0),
                            gridspec_kw={"height_ratios": [0.95, 1.9, 0.85]},
                            constrained_layout=True)
    for j, (nm, p) in enumerate(ARRAYS.items()):
        outer, ctr = p[:-1], p[-1]
        axl = axs[0, j]
        axl.plot(np.append(outer[:, 0], outer[0, 0]),
                 np.append(outer[:, 1], outer[0, 1]), "-", color="0.7", lw=1.0)
        axl.scatter(outer[:, 0], outer[:, 1], s=70, color="#2b6cb0", zorder=3,
                    label="outer samples")
        axl.scatter(ctr[0], ctr[1], s=70, marker="s", color="#c05621", zorder=3,
                    label="center sample")
        th = np.linspace(0, 2 * np.pi, 200)
        axl.plot(CIRC[nm]*np.cos(th), CIRC[nm]*np.sin(th), ":",
                 color="0.75", lw=0.9)
        axl.axhline(0, color="0.92", lw=0.7, zorder=0)
        axl.axvline(0, color="0.92", lw=0.7, zorder=0)
        axl.set_xlim(-18, 18); axl.set_ylim(-18, 18)
        axl.set_aspect("equal", adjustable="box")
        axl.set_xlabel("east [km]", fontsize=8)
        if j == 0:
            axl.set_ylabel("north [km]", fontsize=8)
            axl.legend(fontsize=6.5, loc="upper left", framealpha=0.9)
        axl.tick_params(labelsize=7)
        axl.set_title(f"{nm}\n({LABEL[nm]})", fontsize=9.5)

        Rm = R[nm][2 + idx]
        ax = axs[1, j]
        im = ax.pcolormesh(FX_C, FY_C, Rm, vmin=0, vmax=1,
                           cmap="viridis", shading="auto", rasterized=True)
        ax.contour(FX_C, FY_C, Rm, [0.5], colors="w", linewidths=1.4) # halfpower contour
        ax.contour(FX_C, FY_C, Rm, [0.1], colors="w", linewidths=0.6, # 10% power
                   linestyles=":")
        if Rm.max() > 1.01:
            ax.contour(FX_C, FY_C, Rm, [1.0], colors="r", linewidths=1.0)
            ax.text(0.03, 0.03, f"max R = {Rm.max():.0f}", color="r",
                    transform=ax.transAxes, fontsize=7.5, va="bottom")
        ax.set_xlabel("k [cyc km$^{-1}$]", fontsize=8)
        if j == 0:
            ax.set_ylabel("l [cyc km$^{-1}$]", fontsize=8)
        ax.set_aspect("equal")
        ax.xaxis.set_major_locator(plt.MaxNLocator(3))
        ax.yaxis.set_major_locator(plt.MaxNLocator(3))
        ax.tick_params(labelsize=7.5)
        if j == n - 1:
            fig.colorbar(im, ax=axs[1, :], shrink=0.85, extend="max",
                         label="R  (normalized)")

        axc = axs[2, j]
        ck = Rm[np.argmin(np.abs(FY_C)), :]
        cl = Rm[:, np.argmin(np.abs(FX_C))]
        axc.plot(FX_C, ck, "k", lw=1.3, label="along $k$ ($l=0$)")
        axc.plot(FY_C, cl, color="#c05621", lw=1.3, ls="--",
                 label="along $l$ ($k=0$)")
        axc.axhline(0.5, color="r", lw=0.8, ls=":")
        for f_, c_, col, dy in ((FX_C, ck, "k", 13),
                                (FY_C, cl, "#c05621", -15)):
            m = f_ > 0
            fp, cp = f_[m], c_[m]
            if (cp < 0.5).any():
                hp = fp[cp < 0.5][0]
                axc.plot(hp, 0.5, "o", ms=4.5, color=col)
                axc.annotate(f"{1/hp:.0f} km", (hp, 0.5), color=col, fontsize=7.5,
                             xytext=(5, dy), textcoords="offset points")
        axc.set_xlim(0, FX_C.max()); axc.set_ylim(0, 1.05)
        axc.set_xlabel("wavenumber [cyc km$^{-1}$]", fontsize=8)
        if j == 0:
            axc.set_ylabel("R", fontsize=8); axc.legend(fontsize=6.5)
        axc.tick_params(labelsize=7.5); axc.grid(alpha=0.25)

    d = r"\partial/\partial " + comp
    fig.suptitle(rf"Normalized power transfer $R_{{{comp}}}$ for ${d}$   "
                 r"(white solid = half power, dotted = $R$ = 0.1; "
                 r"red = $R$ = 1 where exceeded)"
                 "\n"
                 rf"equal aperture, circumradius {B_KM:.1f} km, "
                 rf"centered on 0$\degree$N 140$\degree$W",
                 fontsize=12)
    fig.savefig(fname, dpi=130)

In [10]:
comp_fig(0, "x", f"test_dudx.png")
comp_fig(1, "y", f"test_dudy.png")